# Delete LDS configurations #

This is a Jupyter notebook to delete all the LDS configs. 
LDS is going to be decommissioned at the end of June 2026, and customers should move to Datastream. EOL announcement can be found [here](https://techdocs.akamai.com/log-delivery/docs/log-deliv-service-ov).

## Import dependencies and setup EdgeGrid session

Import the EdgeGrid and requests libraries needed to authenticate and call Akamai APIs. 

Make sure to set the correct `~/.edgegrid` section via `EXPORT AKAMAI_EDGEGRID_SECTION=section`  



In [1]:
import os
from pathlib import Path

import requests
from akamai.edgegrid import EdgeGridAuth, EdgeRc

# Use credentials from the gss section in ~/.edgerc.
edgerc_path = os.path.expanduser("~/.edgerc")

if not os.path.exists(edgerc_path):
    raise FileNotFoundError(f"Could not find .edgerc at {edgerc_path}")

# set correct section, default by default
section = os.getenv("AKAMAI_EDGEGRID_SECTION", "default")

# Optional: include accountSwitchKey only when provided.
account_switch_key = os.getenv("AKAMAI_ACCOUNT_SWITCH_KEY")

edgerc = EdgeRc(edgerc_path)
host = edgerc.get(section, "host")

session = requests.Session()
session.auth = EdgeGridAuth.from_edgerc(edgerc, section)
session.headers.update({
    "Accept": "application/json",
    "Content-Type": "application/json",
})

base_url = f"https://{host}"
print(f"EdgeGrid session configured for host: {host}")

EdgeGrid session configured for host: akab-jdn2ct6ewvfefhhb-csofinvorrtwiu5w.luna.akamaiapis.net


## Get list of LDS config id's ##

If we want to delete LDS configs, we need to get the id's of all [LDS configurations](https://techdocs.akamai.com/log-delivery/reference/get-log-configurations). LDS configurations are split by log source type with the following options `[cpcode-products, gtm, edns, answerx, etp]`  
Let's give it a try with EdgeDNS as an example

In [2]:
log_source_types = ["edns", "cpcode-products", "gtm", "answerx", "etp"]
config_ids = []

for source_type in log_source_types:
    url = f"{base_url}/lds-api/v3/log-sources/{source_type}/log-configurations"

    params = {}
    if account_switch_key:
        params["accountSwitchKey"] = account_switch_key

    response = session.get(url, params=params, timeout=120)

    if response.status_code == 200:
        configs = response.json()
        ids = [c["id"] for c in configs]
        config_ids.extend(ids)
        print(f"{source_type}: {len(ids)} configuration(s)")
    else:
        print(f"{source_type}: {response.status_code} - {response.text}")

print(f"\nTotal configs to delete: {len(config_ids)}")
print(config_ids)

edns: 382 configuration(s)
cpcode-products: 15 configuration(s)
gtm: 0 configuration(s)
answerx: 0 configuration(s)
etp: 0 configuration(s)

Total configs to delete: 397
[947215, 947212, 947213, 947210, 947211, 947208, 947209, 947206, 947207, 947204, 947205, 947202, 947203, 947200, 947201, 947230, 947231, 947228, 947229, 947226, 947227, 947224, 947225, 947222, 947223, 947220, 947221, 947218, 947219, 947216, 947217, 947246, 947247, 947244, 947245, 947242, 947243, 947240, 947241, 947238, 947239, 947236, 947237, 947234, 947235, 947232, 947233, 947262, 947263, 947260, 947261, 947258, 947259, 947257, 947254, 947255, 947252, 947253, 947250, 947251, 947248, 947249, 947278, 947279, 947276, 947277, 947274, 947275, 947272, 947273, 947270, 947271, 947268, 947269, 947266, 947267, 947264, 947265, 947294, 947295, 947292, 947293, 947290, 947291, 947288, 947289, 947286, 947287, 947284, 947285, 947282, 947283, 947280, 947281, 947306, 947305, 947302, 947303, 947300, 1044068, 947301, 947298, 947299, 9472

## Delete the LDS configs

Now that we have a list of LDS configs, let's delete them one by one via the delete [API call](https://techdocs.akamai.com/log-delivery/reference/delete-log-configuration).

Manually update last cell to delete all available id's
**As this call will delete all LDS configs**  

In [11]:
# Safety check: only deletes the first config by default.
# Change config_ids[:1] to config_ids to delete all.
for config_id in config_ids[:1]:
    url = f"{base_url}/lds-api/v3/log-configurations/{config_id}"

    params = {}
    if account_switch_key:
        params["accountSwitchKey"] = account_switch_key

    response = session.delete(url, params=params, timeout=120)

    if response.status_code == 204:
        print(f"Deleted {config_id}")
    else:
        print(f"Failed {config_id}: {response.status_code} - {response.text}")

Deleted 707034
